# Guide First Frame Real Centroid Review

这个 notebook 用来审阅 `Real centroid extraction: weighted centroid + exact et_focalplane geometry` 的首帧导星联合解算结果。

这次统一采用两套明确分开的口径：
- `current_to_frame_truth`：当前解相对于“实际仿真帧真值”的姿态误差，这是当前最应该盯的主指标。
- `frame_truth_to_nominal_body`：实际仿真帧相对于名义 `et_focalplane/body` 几何的固定偏移，它主要反映 simulator 的 telescope FOV offset，而不是质心噪声。

建议阅读顺序：
1. 先看“推荐口径”和“高层结论”，确认这次该看哪组指标。
2. 再看“反事实姿态解算”和“telescope FOV offset 证据链”，理解误差来自哪里。
3. 最后看分探测器和逐星 dataframe，定位具体异常星或异常探测器。


## 1. 读取结果文件

这一节只做三件事：
- 读取主结果 JSON
- 读取详细误差审计 JSON
- 顺手读取首个 batch 的 `run_meta.json`，方便核对 `field_offset` 和 detector truth 口径


In [1]:
# 这格统一准备后面所有表格会用到的对象。
# 约定：
# - result: 主结果摘要 JSON
# - audit: 逐星详细审计 JSON
# - summary: audit 的全局统计摘要
# - counter: 反事实姿态解算对比块
# - per_star_df: 逐星 dataframe

from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

ROOT = Path("/home/cxgao/ET/FSG/fsglib")
RESULT_PATH = ROOT / "outputs/debug/guide_first_frame_v1_noise_psf_result.json"
AUDIT_PATH = ROOT / "outputs/debug/guide_first_frame_v1_noise_psf_error_audit.json"

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
summary = result["error_audit"]["summary"]
counter = summary["counterfactual_solutions"]
per_star_df = pd.json_normalize(audit["per_star"])

dataset_root = Path(result["meta"]["dataset_root"])
first_detector = sorted(result["detector_stats"])[0]
first_batch_name = result["detector_stats"][first_detector]["batch_name"]
run_meta_path = dataset_root / first_batch_name / "run_meta.json"
run_meta = json.loads(run_meta_path.read_text(encoding="utf-8"))

print(f"Scenario   : Real centroid extraction: weighted centroid + exact et_focalplane geometry")
print(f"Result JSON: {RESULT_PATH}")
print(f"Audit  JSON: {AUDIT_PATH}")
print(f"Run meta   : {run_meta_path}")
print(f"Per-star rows: {len(per_star_df)}")


Scenario   : Real centroid extraction: weighted centroid + exact et_focalplane geometry
Result JSON: /home/cxgao/ET/FSG/fsglib/outputs/debug/guide_first_frame_v1_noise_psf_result.json
Audit  JSON: /home/cxgao/ET/FSG/fsglib/outputs/debug/guide_first_frame_v1_noise_psf_error_audit.json
Run meta   : /home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch3_ra305.0356_dec38.4755/run_meta.json
Per-star rows: 390


## 2. 推荐口径

这一步最重要，因为现在结果里同时有“当前帧真值口径”和“名义几何口径”。

简单记法：
- 真正汇报“算法在仿真帧上做得怎样”，优先看 `current_to_frame_truth`。
- 如果想解释为什么还存在一个稳定的几何偏差，再看 `frame_truth_to_nominal_body`。
- `predicted_vs_ecsv_detector` 近似为 0，说明 `et_focalplane` 的 raw detector 坐标本身是对的。
- `predicted_vs_truth` 的常量偏移来自 NPZ detector truth 里额外带入的 telescope FOV offset。


In [2]:
# 这张表是 notebook 里最先该统一认知的一张表。
metric_definition_df = pd.DataFrame([
    {
        "metric": "current_to_frame_truth_arcsec",
        "recommended": True,
        "meaning": "当前解相对于实际仿真帧真值的姿态误差，当前最推荐的主指标。",
    },
    {
        "metric": "frame_truth_to_nominal_body_arcsec",
        "recommended": True,
        "meaning": "实际仿真帧相对于名义 et_focalplane/body 几何的固定偏移，主要反映 simulator telescope FOV offset。",
    },
    {
        "metric": "match_predicted_vs_ecsv_detector_pix",
        "recommended": True,
        "meaning": "参考星预测像点相对于 raw detector 坐标的偏差；若接近 0，说明 et_focalplane raw detector 几何是自洽的。",
    },
    {
        "metric": "match_predicted_vs_truth_pix",
        "recommended": False,
        "meaning": "参考星预测像点相对于 NPZ detector truth 的偏差；当前数据里它会混入 telescope FOV offset。",
    },
    {
        "metric": "delta_current_to_oracle_arcsec",
        "recommended": False,
        "meaning": "历史口径，当前不再作为首选主指标，因为它会把 frame truth 和 nominal body 混在一起。",
    },
])
display(metric_definition_df)

interpretation_df = pd.DataFrame(
    [{"key": key, "meaning": value} for key, value in summary.get("interpretation", {}).items()]
)
display(interpretation_df)


,metric,recommended,meaning
0,current_to_frame_truth_arcsec,True,当前解相对于实际仿真帧真值的姿态误差，当前最推荐的主指标。
1,frame_truth_to_nominal_body_arcsec,True,实际仿真帧相对于名义 et_focalplane/body 几何的固定偏移，主要反映 simulator telescope FOV offset。
2,match_predicted_vs_ecsv_detector_pix,True,参考星预测像点相对于 raw detector 坐标的偏差；若接近 0，说明 et_focalplane raw detector 几何是自洽的。
3,match_predicted_vs_truth_pix,False,参考星预测像点相对于 NPZ detector truth 的偏差；当前数据里它会混入 telescope FOV offset。
4,delta_current_to_oracle_arcsec,False,历史口径，当前不再作为首选主指标，因为它会把 frame truth 和 nominal body 混在一起。


,key,meaning
0,match_predicted_vs_truth_pix,Reference predicted_xy compared against NPZ detector truth. In this dataset NPZ detector truth includes the simulator telescope FOV offset.
1,match_predicted_vs_ecsv_detector_pix,Reference predicted_xy compared against raw et_focalplane detector coordinates from stars.ecsv.
2,frame_truth_reference,Use counterfactual_solutions.frame_truth_same_matches and delta_current_to_frame_truth_arcsec as the primary simulated-frame attitude accuracy metric.


## 3. 高层结论

这一节把最关键的数压缩成一张表：
- 姿态是否有效
- 残差 RMS / MAX
- 质心 RMS
- 当前解到“实际仿真帧真值”的姿态差
- 实际仿真帧到“名义几何”的固定偏移
- raw detector 与 NPZ detector truth 的两套坐标差异

这张表最适合直接截图或放进报告首页。


In [12]:
# 这里把最关键的指标都收敛到一张表里。
current_to_frame_truth = counter["delta_components"]["current_to_frame_truth"]
frame_truth_to_nominal = counter["delta_components"]["frame_truth_to_nominal_body"]

overview = pd.DataFrame([
    {"section": "scenario", "metric": "label", "value": "Real centroid extraction: weighted centroid + exact et_focalplane geometry"},
    {"section": "attitude", "metric": "valid", "value": result["solution"]["valid"]},
    {"section": "attitude", "metric": "num_matched", "value": result["solution"]["num_matched"]},
    {"section": "attitude", "metric": "residual_rms_arcsec", "value": result["solution"]["residual_rms_arcsec"]},
    {"section": "attitude", "metric": "residual_max_arcsec", "value": result["solution"]["residual_max_arcsec"]},
    {"section": "matching", "metric": "mean_residual_pix", "value": result["matching"]["debug"]["mean_residual_pix"]},
    {"section": "centroid", "metric": "detector_rms_pix", "value": summary["centroid_error_detector_pix"]["rms_radial"]},
    {"section": "body", "metric": "body_total_rms_arcsec", "value": summary["body_error_total_arcsec"]["rms"]},
    {"section": "recommended", "metric": "current_to_frame_truth_total_arcsec", "value": current_to_frame_truth["total_arcsec"]},
    {"section": "recommended", "metric": "current_to_frame_truth_non_roll_arcsec", "value": current_to_frame_truth["non_roll_arcsec"]},
    {"section": "recommended", "metric": "current_to_frame_truth_roll_arcsec", "value": current_to_frame_truth["roll_arcsec"]},
    {"section": "nominal_gap", "metric": "frame_truth_to_nominal_total_arcsec", "value": frame_truth_to_nominal["total_arcsec"]},
    {"section": "nominal_gap", "metric": "frame_truth_to_nominal_non_roll_arcsec", "value": frame_truth_to_nominal["non_roll_arcsec"]},
    {"section": "nominal_gap", "metric": "frame_truth_to_nominal_roll_arcsec", "value": frame_truth_to_nominal["roll_arcsec"]},
    {"section": "detector_coords", "metric": "predicted_vs_raw_detector_rms_pix", "value": summary["match_predicted_vs_ecsv_detector_pix"]["rms_radial"]},
    {"section": "detector_coords", "metric": "npz_minus_raw_detector_offset_rms_pix", "value": summary["npz_minus_ecsv_detector_offset_pix"]["rms_radial"]},
])
display(overview)

quaternion_df = pd.DataFrame([
    {
        "q_w": result["solution"]["q_ib"][0],
        "q_x": result["solution"]["q_ib"][1],
        "q_y": result["solution"]["q_ib"][2],
        "q_z": result["solution"]["q_ib"][3],
    }
])
display(quaternion_df)


,section,metric,value
0,scenario,label,Real centroid extraction: weighted centroid + exact et_focalplane geometry
1,attitude,valid,True
2,attitude,num_matched,390
3,attitude,residual_rms_arcsec,1.803370
4,attitude,residual_max_arcsec,4.119148
5,matching,mean_residual_pix,0.490234
6,centroid,detector_rms_pix,0.651399
7,body,body_total_rms_arcsec,1.681581
8,recommended,current_to_frame_truth_total_arcsec,0.515328
9,recommended,current_to_frame_truth_non_roll_arcsec,0.485685


,q_w,q_x,q_y,q_z
0,0.685905,-0.320306,0.130819,-0.640175


## 4. 反事实姿态解算

这一节是误差传播的主表。

四套解的推荐理解：
- `current`：当前真实链路结果
- `frame_truth_same_matches`：把观测 LOS 换成“实际仿真帧真值”，但保留当前匹配
- `nominal_body_same_matches`：把观测 LOS 换成名义 `et_focalplane/body` 几何
- `oracle_truth_body_and_match`：真值 body + 真值匹配，作为极限参考

在 exact 几何链路下：
- `current -> frame_truth` 更像真实算法误差
- `frame_truth -> nominal_body` 更像 simulator 与 nominal geometry 的口径差


In [4]:
# 先看四套姿态解本身的质量指标。
name_map = {
    "current": "current",
    "frame_truth_same_matches": "frame_truth_same_matches",
    "truth_pixel_same_matches": "truth_pixel_same_matches (legacy alias)",
    "nominal_body_same_matches": "nominal_body_same_matches",
    "exact_body_same_matches": "exact_body_same_matches (legacy alias)",
    "oracle_truth_body_and_match": "oracle_truth_body_and_match",
}

counter_rows = []
for key in [
    "current",
    "frame_truth_same_matches",
    "nominal_body_same_matches",
    "oracle_truth_body_and_match",
]:
    payload = counter.get(key)
    if payload is None:
        continue
    counter_rows.append({
        "solution_name": name_map.get(key, key),
        "valid": payload["valid"],
        "num_matched": payload["num_matched"],
        "num_rejected": payload["num_rejected"],
        "residual_rms_arcsec": payload["residual_rms_arcsec"],
        "residual_max_arcsec": payload["residual_max_arcsec"],
        "quality_flag": payload["quality_flag"],
        "degraded_level": payload["degraded_level"],
    })

display(pd.DataFrame(counter_rows))


,solution_name,valid,num_matched,num_rejected,residual_rms_arcsec,residual_max_arcsec,quality_flag,degraded_level
0,current,True,390,0,1.803370,4.119148,VALID,NORMAL_4D
1,frame_truth_same_matches,True,390,0,0.010009,0.012673,VALID,NORMAL_4D
2,nominal_body_same_matches,True,390,0,0.000220,0.003074,VALID,NORMAL_4D
3,oracle_truth_body_and_match,True,390,0,0.000220,0.003074,VALID,NORMAL_4D


In [5]:
# 再看姿态差如何分解成 total / non-roll / roll。
delta_components_df = pd.DataFrame([
    {"delta_name": key, **value}
    for key, value in counter["delta_components"].items()
])
display(delta_components_df)


,delta_name,non_roll_arcsec,roll_arcsec,total_arcsec
0,current_to_frame_truth,0.485685,0.172250,0.515328
1,current_to_truth_pixel,0.485685,0.172250,0.515328
2,frame_truth_to_nominal_body,0.431738,0.003964,0.431766
3,truth_pixel_to_exact_body,0.431738,0.003964,0.431766
4,current_to_nominal_body,0.295128,0.168285,0.339761
5,current_to_exact_body,0.295128,0.168285,0.339761
6,current_to_oracle,0.295128,0.168285,0.339761
7,exact_body_to_oracle_match,0.000000,0.000000,0.000000


## 5. telescope FOV offset 证据链

这里专门回答一个容易混淆的问题：

为什么 `run_meta` 里脚本层 `field_offset_x/y = 0`，但 `NPZ detector truth` 和 `stars.ecsv raw detector` 之间仍然差了一个固定常量？

当前证据链是：
1. `run_meta` 只记录脚本层 static field offset，并没有写 telescope 级随机偏移。
2. `predicted_vs_raw_detector` 近似 0，说明 `et_focalplane` raw detector 坐标没问题。
3. `predicted_vs_truth` 与 `npz_minus_raw_detector_offset` 相同，说明这个偏移来自 NPZ detector truth 的定义，而不是匹配器。


In [6]:
# 先看 run_meta 里能直接读到的 offset 相关字段。
run_meta_offset_df = pd.DataFrame([{
    "apply_static_field_offset": run_meta.get("apply_static_field_offset"),
    "field_offset_x_pix": run_meta.get("field_offset_x_pix"),
    "field_offset_y_pix": run_meta.get("field_offset_y_pix"),
    "requested_field_offset_x_pix": run_meta.get("requested_field_offset_x_pix"),
    "requested_field_offset_y_pix": run_meta.get("requested_field_offset_y_pix"),
    "guide_query_target_center_xpix": run_meta.get("guide_query_target_center_xpix"),
    "guide_query_target_center_ypix": run_meta.get("guide_query_target_center_ypix"),
}])
display(run_meta_offset_df)

# 再看这次真正影响理解的几项证据。
evidence_df = pd.DataFrame([
    {"metric": "sim_to_detector_map_rms_pix", "value": summary["sim_to_detector_map_error_pix"]["rms_radial"]},
    {"metric": "predicted_vs_raw_detector_rms_pix", "value": summary["match_predicted_vs_ecsv_detector_pix"]["rms_radial"]},
    {"metric": "predicted_vs_npz_truth_rms_pix", "value": summary["match_predicted_vs_truth_pix"]["rms_radial"]},
    {"metric": "npz_minus_raw_detector_offset_rms_pix", "value": summary["npz_minus_ecsv_detector_offset_pix"]["rms_radial"]},
])
display(evidence_df)

per_detector_offset_rows = []
for detector_id, payload in audit["per_detector"].items():
    per_detector_offset_rows.append({
        "detector_id": detector_id,
        "predicted_vs_raw_detector_rms_pix": payload["match_predicted_vs_ecsv_detector_pix"]["rms_radial"],
        "predicted_vs_npz_truth_rms_pix": payload["match_predicted_vs_truth_pix"]["rms_radial"],
        "npz_minus_raw_detector_offset_rms_pix": payload["npz_minus_ecsv_detector_offset_pix"]["rms_radial"],
        "npz_minus_raw_detector_mean_dx_pix": payload["npz_minus_ecsv_detector_offset_pix"]["mean_dx"],
        "npz_minus_raw_detector_mean_dy_pix": payload["npz_minus_ecsv_detector_offset_pix"]["mean_dy"],
    })
display(pd.DataFrame(per_detector_offset_rows).sort_values("detector_id"))


,apply_static_field_offset,field_offset_x_pix,field_offset_y_pix,requested_field_offset_x_pix,requested_field_offset_y_pix,guide_query_target_center_xpix,guide_query_target_center_ypix
0,False,0.000000,0.000000,None,None,1024.849292,1024.583885


,metric,value
0,sim_to_detector_map_rms_pix,0.000000
1,predicted_vs_raw_detector_rms_pix,0.000000
2,predicted_vs_npz_truth_rms_pix,0.141759
3,npz_minus_raw_detector_offset_rms_pix,0.141759


,detector_id,predicted_vs_raw_detector_rms_pix,predicted_vs_npz_truth_rms_pix,npz_minus_raw_detector_offset_rms_pix,npz_minus_raw_detector_mean_dx_pix,npz_minus_raw_detector_mean_dy_pix
3,guide_bottom,0.000000,0.141759,0.141759,0.132633,-0.050039
0,guide_left,0.000000,0.141759,0.141759,0.132633,-0.050039
2,guide_right,0.000000,0.141759,0.141759,0.132633,-0.050039
1,guide_top,0.000000,0.141759,0.141759,0.132633,-0.050039


## 6. 分探测器链路通过情况

这一节先只看数量：
- 每片 detector 选了多少星
- 最终匹配了多少星
- 参考星有多少

如果这里已经不通，后面的精度表就没有解释意义。


In [7]:
detector_counts_rows = []
for detector_id, stats in result["detector_stats"].items():
    row = {"detector_id": detector_id, **stats}
    detector_counts_rows.append(row)

detector_counts_df = pd.DataFrame(detector_counts_rows).sort_values("detector_id")
display(detector_counts_df)


,detector_id,batch_name,frame_path,num_candidates_raw,num_candidates_selected,num_matched,sim_to_detector_kind,schema_version,offset_x_pix,offset_y_pix,num_reference_stars
3,guide_bottom,batch3_ra305.0356_dec38.4755,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch3_ra305.0356_dec38.4755/frames/scope0_coadd_000000_000000.npz,95,95,95,offset,2,51.849292,51.583885,260
0,guide_left,batch0_ra278.1844_dec37.5704,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch0_ra278.1844_dec37.5704/frames/scope0_coadd_000000_000000.npz,95,95,95,offset,2,51.592127,51.384965,260
2,guide_right,batch2_ra310.6239_dec59.2676,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch2_ra310.6239_dec59.2676/frames/scope0_coadd_000000_000000.npz,153,100,100,offset,2,50.108021,51.848764,260
1,guide_top,batch1_ra269.5821_dec57.8997,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch1_ra269.5821_dec57.8997/frames/scope0_coadd_000000_000000.npz,127,100,100,offset,2,51.380730,50.110761,220


## 7. 分探测器关键精度

这一节横向比较四片导星的关键误差：
- 质心误差
- raw detector / NPZ detector truth 的两套口径差异
- `body` 方向误差
- 像面匹配残差

如果某片明显异常，通常从这里最容易先看出来。


In [8]:
detector_metric_rows = []
for detector_id, payload in audit["per_detector"].items():
    detector_metric_rows.append({
        "detector_id": detector_id,
        "centroid_rms_pix": payload["centroid_error_detector_pix"]["rms_radial"],
        "body_centroid_rms_arcsec": payload["body_error_centroid_arcsec"]["rms"],
        "body_geometry_rms_arcsec": payload["body_error_geometry_arcsec"]["rms"],
        "body_total_rms_arcsec": payload["body_error_total_arcsec"]["rms"],
        "match_obs_pred_rms_pix": payload["match_observed_vs_predicted_pix"]["rms"],
        "predicted_vs_raw_detector_rms_pix": payload["match_predicted_vs_ecsv_detector_pix"]["rms_radial"],
        "predicted_vs_npz_truth_rms_pix": payload["match_predicted_vs_truth_pix"]["rms_radial"],
        "npz_minus_raw_detector_offset_rms_pix": payload["npz_minus_ecsv_detector_offset_pix"]["rms_radial"],
        "solution_model_rms_arcsec": payload["solution_residual_model_arcsec"]["rms"],
        "solution_exact_body_rms_arcsec": payload["solution_residual_exact_body_arcsec"]["rms"],
    })

detector_metric_df = pd.DataFrame(detector_metric_rows).sort_values("detector_id")
display(detector_metric_df)


,detector_id,centroid_rms_pix,body_centroid_rms_arcsec,body_geometry_rms_arcsec,body_total_rms_arcsec,match_obs_pred_rms_pix,predicted_vs_raw_detector_rms_pix,predicted_vs_npz_truth_rms_pix,npz_minus_raw_detector_offset_rms_pix,solution_model_rms_arcsec,solution_exact_body_rms_arcsec
3,guide_bottom,0.693941,2.091516,0.427782,1.791101,0.594669,0.000000,0.141759,0.141759,1.959064,0.306705
0,guide_left,0.763895,2.278838,0.420902,1.982747,0.663848,0.000000,0.141759,0.141759,2.108306,0.242892
2,guide_right,0.513333,1.533983,0.422852,1.320677,0.441356,0.000000,0.141759,0.141759,1.423796,0.328171
1,guide_top,0.616041,1.855354,0.428802,1.581626,0.525947,0.000000,0.141759,0.141759,1.669589,0.289293


## 8. 逐星 dataframe

这一节把详细审计 JSON 展平成 `per_star_df`。
后面如果要抓最差的星、检查错配、看单颗星误差传播，都从这张表开始。


In [9]:
print("per_star_df shape:", per_star_df.shape)

focus_cols = [
    "detector_id",
    "observed_source_id",
    "truth_index",
    "snr",
    "centroid_error_detector_radial_pix",
    "body_error_total_arcsec",
    "match_residual_pix",
    "predicted_vs_ecsv_detector_radial_pix",
    "predicted_vs_truth_radial_pix",
    "match_is_correct",
]
display(per_star_df[focus_cols].head(10))


per_star_df shape: (390, 67)


,detector_id,observed_source_id,truth_index,snr,centroid_error_detector_radial_pix,body_error_total_arcsec,match_residual_pix,predicted_vs_ecsv_detector_radial_pix,predicted_vs_truth_radial_pix,match_is_correct
0,guide_left,guide_left:0,79,16.793265,1.058214,2.809897,0.939715,0.000000,0.141759,True
1,guide_left,guide_left:1,22,44.666520,0.315303,0.603449,0.200772,0.000000,0.141759,True
2,guide_left,guide_left:2,6,149.540751,0.286511,1.075816,0.359096,0.000000,0.141759,True
3,guide_left,guide_left:3,56,22.884812,0.693901,1.712597,0.571171,0.000000,0.141759,True
4,guide_left,guide_left:4,21,48.444529,0.343282,0.725711,0.241208,0.000000,0.141759,True
5,guide_left,guide_left:5,82,14.692796,1.213685,3.237908,1.085979,0.000000,0.141759,True
6,guide_left,guide_left:6,71,16.370951,1.051243,2.740585,0.918688,0.000000,0.141759,True
7,guide_left,guide_left:7,53,24.358570,0.682230,1.646375,0.552153,0.000000,0.141759,True
8,guide_left,guide_left:8,32,36.397657,0.539288,1.207015,0.405403,0.000000,0.141759,True
9,guide_left,guide_left:9,50,25.930962,0.800289,2.007501,0.674038,0.000000,0.141759,True


## 9. 最大误差星表

下面几张表分别抓出：
- 质心最差的星
- `body` 总误差最大的星
- 像面匹配残差最大的星
- detector truth 口径差最显著的星


In [10]:
centroid_cols = [
    "detector_id", "observed_source_id", "truth_index", "snr",
    "centroid_error_detector_radial_pix", "body_error_centroid_arcsec", "body_error_total_arcsec",
]
body_cols = [
    "detector_id", "observed_source_id", "truth_index", "snr",
    "body_error_centroid_arcsec", "body_error_geometry_arcsec", "body_error_total_arcsec",
    "solution_residual_model_arcsec", "solution_residual_exact_body_arcsec",
]
match_cols = [
    "detector_id", "observed_source_id", "truth_index", "snr",
    "match_residual_pix", "predicted_vs_ecsv_detector_radial_pix", "predicted_vs_truth_radial_pix",
    "matched_catalog_id", "truth_catalog_source_id", "match_is_correct",
]

display(per_star_df[centroid_cols].sort_values("centroid_error_detector_radial_pix", ascending=False).head(20))
display(per_star_df[body_cols].sort_values("body_error_total_arcsec", ascending=False).head(20))
display(per_star_df[match_cols].sort_values("match_residual_pix", ascending=False).head(20))
display(per_star_df[match_cols].sort_values("predicted_vs_truth_radial_pix", ascending=False).head(20))


,detector_id,observed_source_id,truth_index,snr,centroid_error_detector_radial_pix,body_error_centroid_arcsec,body_error_total_arcsec
258,guide_right,guide_right:21,86,33.418292,1.281522,3.862323,3.790114
34,guide_left,guide_left:34,104,11.879753,1.273786,3.796576,3.446994
5,guide_left,guide_left:5,82,14.692796,1.213685,3.616777,3.237908
329,guide_bottom,guide_bottom:34,97,11.249583,1.210103,3.643266,3.270263
369,guide_bottom,guide_bottom:74,108,10.956924,1.203545,3.637283,3.241757
60,guide_left,guide_left:60,76,15.695089,1.199750,3.581132,3.228216
112,guide_top,guide_top:77,35,67.499075,1.191788,3.563580,3.305423
187,guide_top,guide_top:95,94,18.215637,1.179460,3.551280,3.190966
79,guide_left,guide_left:79,102,12.790858,1.176433,3.498839,3.120646
93,guide_left,guide_left:93,91,12.404613,1.166443,3.472436,3.115718


,detector_id,observed_source_id,truth_index,snr,body_error_centroid_arcsec,body_error_geometry_arcsec,body_error_total_arcsec,solution_residual_model_arcsec,solution_residual_exact_body_arcsec
258,guide_right,guide_right:21,86,33.418292,3.862323,0.421552,3.790114,4.119152,0.329046
34,guide_left,guide_left:34,104,11.879753,3.796576,0.421014,3.446994,3.628343,0.241604
112,guide_top,guide_top:77,35,67.499075,3.563580,0.429203,3.305423,3.070359,0.287787
329,guide_bottom,guide_bottom:34,97,11.249583,3.643266,0.427648,3.270263,3.501785,0.306943
369,guide_bottom,guide_bottom:74,108,10.956924,3.637283,0.429742,3.241757,3.447299,0.306281
5,guide_left,guide_left:5,82,14.692796,3.616777,0.421429,3.237908,3.397780,0.243900
60,guide_left,guide_left:60,76,15.695089,3.581132,0.420228,3.228216,3.406703,0.242346
187,guide_top,guide_top:95,94,18.215637,3.551280,0.428310,3.190966,3.360784,0.286603
193,guide_top,guide_top:106,95,17.119292,3.478834,0.428200,3.126061,3.304559,0.289766
79,guide_left,guide_left:79,102,12.790858,3.498839,0.420194,3.120646,3.275017,0.239740


,detector_id,observed_source_id,truth_index,snr,match_residual_pix,predicted_vs_ecsv_detector_radial_pix,predicted_vs_truth_radial_pix,matched_catalog_id,truth_catalog_source_id,match_is_correct
258,guide_right,guide_right:21,86,33.418292,1.256473,0.000000,0.141759,2193502783768934144,2193502783768934144,True
34,guide_left,guide_left:34,104,11.879753,1.155611,0.000000,0.141759,2096182260813011712,2096182260813011712,True
112,guide_top,guide_top:77,35,67.499075,1.106884,0.000000,0.141759,1421699497134929408,1421699497134929408,True
329,guide_bottom,guide_bottom:34,97,11.249583,1.086835,0.000000,0.141759,2061080863197553024,2061080863197553024,True
5,guide_left,guide_left:5,82,14.692796,1.085979,0.000000,0.141759,2096083579645383296,2096083579645383296,True
60,guide_left,guide_left:60,76,15.695089,1.080393,0.000000,0.141759,2096260669736367232,2096260669736367232,True
369,guide_bottom,guide_bottom:74,108,10.956924,1.073200,0.000000,0.141759,2061310596723770752,2061310596723770752,True
187,guide_top,guide_top:95,94,18.215637,1.060852,0.000000,0.141759,1421621878485891456,1421621878485891456,True
79,guide_left,guide_left:79,102,12.790858,1.048435,0.000000,0.141759,2096234728134332928,2096234728134332928,True
93,guide_left,guide_left:93,91,12.404613,1.045592,0.000000,0.141759,2095862612167939712,2095862612167939712,True


,detector_id,observed_source_id,truth_index,snr,match_residual_pix,predicted_vs_ecsv_detector_radial_pix,predicted_vs_truth_radial_pix,matched_catalog_id,truth_catalog_source_id,match_is_correct
370,guide_bottom,guide_bottom:75,57,27.072135,0.535290,0.000000,0.141759,2061275579830803584,2061275579830803584,True
280,guide_right,guide_right:40,85,26.229444,0.509770,0.000000,0.141759,2193485260301842432,2193485260301842432,True
287,guide_right,guide_right:43,93,24.370357,0.703292,0.000000,0.141759,2193493231756660864,2193493231756660864,True
86,guide_left,guide_left:86,90,13.323486,1.026403,0.000000,0.141759,2108412712764502528,2108412712764502528,True
20,guide_left,guide_left:20,38,30.569512,0.537275,0.000000,0.141759,2096104779603933440,2096104779603933440,True
351,guide_bottom,guide_bottom:56,1,956.019762,0.489971,0.000000,0.141759,2061242908036996352,2061242908036996352,True
117,guide_top,guide_top:111,23,59.727275,0.129826,0.000000,0.141759,1421877510643827840,1421877510643827840,True
364,guide_bottom,guide_bottom:69,50,30.690313,0.491205,0.000000,0.141759,2061415909308447488,2061415909308447488,True
201,guide_right,guide_right:134,6,337.036208,0.474669,0.000000,0.141759,2193638745252745856,2193638745252745856,True
329,guide_bottom,guide_bottom:34,97,11.249583,1.086835,0.000000,0.141759,2061080863197553024,2061080863197553024,True


## 10. 统计分布

最后一节给全局统计和按 detector 分组统计。
如果你想快速判断“是不是存在明显系统偏差”，这一节最有用。


In [11]:
numeric_cols = [
    "centroid_error_detector_radial_pix",
    "body_error_centroid_arcsec",
    "body_error_geometry_arcsec",
    "body_error_total_arcsec",
    "match_residual_pix",
    "predicted_vs_ecsv_detector_radial_pix",
    "predicted_vs_truth_radial_pix",
    "solution_residual_model_arcsec",
    "solution_residual_exact_body_arcsec",
]

display(per_star_df[numeric_cols].describe().T)
grouped = per_star_df.groupby("detector_id")[numeric_cols].agg(["mean", "median", "max"])
display(grouped)


,count,mean,std,min,25%,50%,75%,max
centroid_error_detector_radial_pix,390.000000,0.580031,0.296834,0.140235,0.318109,0.531692,0.819162,1.281522
body_error_centroid_arcsec,390.000000,1.739708,0.889951,0.420587,0.949253,1.602437,2.442758,3.862323
body_error_geometry_arcsec,390.000000,0.425102,0.003423,0.419114,0.421756,0.425711,0.428219,0.430796
body_error_total_arcsec,390.000000,1.470266,0.817158,0.129675,0.875759,1.344150,2.068200,3.790114
match_residual_pix,390.000000,0.490234,0.272445,0.043263,0.290293,0.448011,0.689384,1.256473
predicted_vs_ecsv_detector_radial_pix,390.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
predicted_vs_truth_radial_pix,390.000000,0.141759,0.000000,0.141759,0.141759,0.141759,0.141759,0.141759
solution_residual_model_arcsec,390.000000,1.552686,0.918422,0.015971,0.829093,1.436208,2.237385,4.119152
solution_residual_exact_body_arcsec,390.000000,0.292196,0.031340,0.238772,0.286281,0.298570,0.326979,0.329720


centroid_error_detector_radial_pix                   body_error_centroid_arcsec                   body_error_geometry_arcsec                   body_error_total_arcsec                   match_residual_pix  \
                                           mean   median      max                       mean   median      max                       mean   median      max                    mean   median      max               mean   
detector_id                                                                                                                                                                                                                
guide_bottom                           0.626699 0.586582 1.210103                   1.888141 1.773440 3.643266                   0.427781 0.427648 0.429742                1.582547 1.428877 3.270263           0.525659   
guide_left                             0.696650 0.693743 1.273786                   2.079209 2.067634 3.796576                   0.420902 0.420834 0.423374                1.794204 1.736633 3.446994           0.600453   
guide_right                            0.463326 0.443638 1.281522                   1.385295 1.326257 3.862323                   0.422850 0.422934 0.425633                1.174332 1.145249 3.790114           0.392425   
guide_top                              0.541615 0.486981 1.191788                   1.630583 1.466208 3.563580                   0.428801 0.428680 0.430796                1.351790 1.243648 3.305423           0.449681   

                               predicted_vs_ecsv_detector_radial_pix                   predicted_vs_truth_radial_pix                   solution_residual_model_arcsec                    \
               median      max                                  mean   median      max                          mean   median      max                           mean   median      max   
detector_id                                                                                                                                                                               
guide_bottom 0.474249 1.086835                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       1.724513 1.588309 3.501785   
guide_left   0.578738 1.155611                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       1.892846 1.934521 3.628343   
guide_right  0.382390 1.256473                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       1.220576 1.222298 4.119152   
guide_top    0.413215 1.106884                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       1.398409 1.301879 3.360784   

             solution_residual_exact_body_arcsec                    
                                            mean   median      max  
detector_id                                                         
guide_bottom                            0.306704 0.306712 0.308693  
guide_left                              0.242884 0.242852 0.247552  
guide_right                             0.328170 0.328097 0.329720  
guide_top                               0.289288 0.289293 0.292249